## Production-Grade Financial RAG with RAGWire (Compare Multiple Companies using Ollama)

Till now, we built simple RAG systems…
But those are not enough for real-world use.

In real applications, you need:
- multiple documents
- metadata filtering
- scalable ingestion

And that’s exactly what we’ll build today using RAGWire — a production-grade RAG framework.

This works because RAGWire understands metadata —
so it can compare Apple vs Google correctly instead of mixing data.

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from ragwire import RAGWire, setup_logging

logger = setup_logging(log_level="INFO")

import ragwire
print(ragwire.__version__)

rag = RAGWire("config.yaml")

1.1.9


In [4]:
# Ingest
stats = rag.ingest_directory("./data")
print(f"Chunks created: {stats['chunks_created']}")

Ingesting: 100%|██████████| 3/3 [00:00<00:00, 46.15file/s]


Chunks created: 0


In [5]:
rag.get_stats()

{'collection_name': 'multi_financial_docs',
 'total_documents': 119,
 'vector_size': 1024,
 'indexed': 119}

In [6]:
rag.discover_metadata_fields()

['source',
 'file_name',
 'file_type',
 'file_hash',
 'chunk_id',
 'chunk_hash',
 'chunk_index',
 'total_chunks',
 'created_at',
 'company_name',
 'doc_type',
 'fiscal_quarter',
 'fiscal_year']

In [7]:
rag.get_field_values(["fiscal_quarter", "doc_type", "fiscal_year", "company_name"])

{'fiscal_quarter': [],
 'doc_type': ['10-k'],
 'fiscal_year': [2025],
 'company_name': ['alphabet inc.', 'amazon.com, inc.', 'apple inc']}

In [8]:
result1 = rag.retrieve("What is the revenue of google?")

# Auto — LLM extracts {"company_name": "apple", "fiscal_year": 2025} from the query
result2 = rag.retrieve("What is Apple's revenue?")

In [9]:
result1

[Document(metadata={'source': 'data\\GOOG-10-K-2025.pdf', 'file_name': 'GOOG-10-K-2025.pdf', 'file_type': 'pdf', 'file_hash': '9953e8d2e70b18c5bf0af3b2b2a1e96de0555f51cbb39a625497b434e19fb43a', 'chunk_id': '9953e8d2e70b18c5bf0af3b2b2a1e96de0555f51cbb39a625497b434e19fb43a_18', 'chunk_hash': '9f0bebd9dbd7cd196a91bcbcc5909442aa6b786e5d10a8ee1d2379be8cf0ddb7', 'chunk_index': 18, 'total_chunks': 46, 'created_at': '2026-03-24T09:08:51.189966+00:00', 'company_name': 'alphabet inc.', 'doc_type': '10-k', 'fiscal_quarter': None, 'fiscal_year': [2025], '_id': 'be522608-6768-484c-9685-019c43637d86', '_collection_name': 'multi_financial_docs'}, page_content='Revenues and Monetization Metrics\n\nWe generate revenues by delivering relevant, cost-effective online advertising; cloud-based solutions that provide\nenterprise  customers  of  all  sizes  with  infrastructure,  platform  services,  and  applications;  and  sales  of  other  products\nand  services,  such  as  fees  received  for  subscripti

In [10]:
result2

[Document(metadata={'source': 'data\\Apple_10k_2025.pdf', 'file_name': 'Apple_10k_2025.pdf', 'file_type': 'pdf', 'file_hash': '108590052c3ba5400c63660d787fe7ed4e43868292946d7a7facebe9ab7d1aab', 'chunk_id': '108590052c3ba5400c63660d787fe7ed4e43868292946d7a7facebe9ab7d1aab_16', 'chunk_hash': '60b8d552baec48866282206f62d1d54dad52bc158891d1dcf5f9849c24cf4082', 'chunk_index': 16, 'total_chunks': 35, 'created_at': '2026-03-24T09:04:44.288328+00:00', 'company_name': 'apple inc', 'doc_type': '10-k', 'fiscal_quarter': None, 'fiscal_year': [2025], '_id': 'c4c7a96f-b510-4448-9e79-f4106c08379e', '_collection_name': 'multi_financial_docs'}, page_content='Earnings per share:\n\nBasic\nDiluted\n\nShares used in computing earnings per share:\n\nBasic\n\nDiluted\n\nSeptember 27,\n2025\n\nYears ended\n\nSeptember 28,\n2024\n\nSeptember 30,\n2023\n\n$\n\n$\n\n$\n$\n\n307,003  $\n109,158\n\n416,161\n\n294,866  $\n96,169\n\n391,035\n\n194,116\n26,844\n\n220,960\n\n195,201\n\n34,550\n27,601\n62,151\n\n185,2

In [11]:
rag.extract_metadata("What is Apple's revenue for 2025?")

{'company_name': 'apple inc.',
 'doc_type': '10-k',
 'fiscal_quarter': None,
 'fiscal_year': [2025]}

In [12]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_ollama import ChatOllama
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_documents(query: str) -> str:
    """Search the document knowledge base for relevant information."""
    results = rag.retrieve(query, top_k=5)
    if not results:
        return "No relevant documents found."
    return "\n\n---\n\n".join(
        f"[{doc.metadata.get('file_name')}]\n{doc.page_content}"
        for doc in results
    )

agent = create_agent(
    model=ChatOllama(model="nemotron-3-nano:4b", temperature=0, base_url="http://192.168.1.9:11434"),
    tools=[search_documents],
    system_prompt="You are a helpful document assistant. Use search_documents to answer questions.",
    checkpointer=InMemorySaver(),
)

In [13]:
# import markdown display
from IPython.display import Markdown, display

config = {"configurable": {"thread_id": "session-1"}}
response = agent.invoke(
    {"messages": [HumanMessage("Compare the revenue of Apple and Google for 2025.")]},
    config=config,
)
Markdown(response["messages"][-1].content)

Based on the 2025 financial data from both companies' 10-K filings:

**Apple's 2025 Revenue:** $416,161 million

**Google's 2025 Revenue:** $402,836 million

**Comparison:** Apple's revenue is $13,325 million higher than Google's revenue in 2025.

This represents a 3.3% difference in revenue between the two companies for 2025.

In [15]:
response = agent.invoke(
    {"messages": [HumanMessage("Compare the cashflow of Apple, Amazon and Google for 2025.")]},
    config=config,
)
Markdown(response["messages"][-1].content)

Based on the 2025 financial data from the 10-K filings, here's a comparison of the cash flow statements for Apple, Amazon, and Google:

## **2025 Cash Flow Comparison**

### **Apple (2025)**
- **Operating Cash Flow**: $112,010 million
- **Investing Cash Flow**: -$118,254 million (used)
- **Financing Cash Flow**: -$121,983 million (used)
- **Net Increase in Cash**: +$1,400 million

### **Amazon (2025)**
- **Operating Cash Flow**: $115,877 million
- **Investing Cash Flow**: -$82,999 million (used)
- **Financing Cash Flow**: -$131,819 million (used)
- **Net Increase in Cash**: +$5,142 million

### **Google (2025)**
- **Operating Cash Flow**: $125,299 million
- **Investing Cash Flow**: -$45,536 million (used)
- **Financing Cash Flow**: -$37,388 million (used)
- **Net Increase in Cash**: +$37,388 million

## **Key Observations:**

1. **Operating Cash Flow**: Google generated the highest operating cash flow at $125.3 billion, followed by Amazon ($115.9 billion) and Apple ($112.0 billion).

2. **Capital Expenditures**: Apple invested the most in infrastructure ($118.3 billion), while Google invested the least ($45.5 billion). This reflects different growth strategies.

3. **Shareholder Returns**: Amazon returned the most cash to shareholders through buybacks and debt repayments ($131.8 billion), while Google returned the least ($37.4 billion).

4. **Cash Position**: Google ended with the largest increase in cash ($37.4 billion), followed by Amazon ($5.1 billion) and Apple ($1.4 billion).

5. **Total Cash Flow**: All three companies had positive net cash flows, indicating healthy financial operations.

**Summary**: Google had the strongest cash flow generation and the largest cash increase, while Apple invested heavily in infrastructure and Amazon returned the most to shareholders.